<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/ml3-time-series-forecasting/blob/main/taller1_pronostico_ocupados.ipynb)

# Pronóstico del número de ocupados en las 13 principales ciudades de Colombia

**Aprendizaje Automático III — Universidad Icesi**

**Integrantes:** Juan Esteban Cardona, Daniel Velasco

Este notebook construye y evalúa modelos de pronóstico para el número mensual de ocupados, en miles de personas, de las 13 principales ciudades de Colombia, con el objetivo de pronosticar los próximos 6 meses. Se comparan tres familias de modelos sobre la misma serie: promedios móviles con distintos tamaños de ventana, suavización exponencial con distintas configuraciones y regresión lineal con distintas formas de representar la tendencia y la estacionalidad.

Todos los modelos se evalúan con el mismo esquema: una validación que respeta el orden temporal de los datos, con las mismas métricas y los mismos cortes, y una muestra de prueba con los 6 meses más recientes, que no se usa para tomar ninguna decisión de modelado.

### Contenido

**Preparación común**

1. **Carga de paquetes:** librerías y versiones del entorno.
2. **Carga y exploración de los datos:** periodo, valores faltantes y comportamiento general de la serie.
3. **Partición de entrenamiento y prueba:** los últimos 6 meses se reservan como prueba.
4. **Componentes de la serie y tipo de estacionalidad:** descomposición de la serie y elección entre estacionalidad aditiva y multiplicativa.
5. **Esquema de validación y modelos de referencia:** validación con origen móvil, métricas y pronósticos naive que todo modelo debe superar.

**Parte I — Promedios móviles y suavización exponencial**

6. **Promedios móviles:** ventanas de distinto tamaño.
7. **Suavización exponencial:** configuraciones de SES, Holt y Holt-Winters, y elección del mejor modelo de esta familia.

**Parte II — Regresión lineal**

8. **Regresión con tendencia:** tendencias polinómicas del tiempo.
9. **Tendencia y estacionalidad con variables dummy:** efectos de mes y revisión de tendencias de mayor grado.
10. **Ventana móvil frente a ventana recursiva:** estimación con los años más recientes.
11. **Estacionalidad con términos de Fourier:** representación del patrón estacional con ondas.
12. **Estacionalidad multiplicativa:** regresión sobre el logaritmo de la serie.
13. **Selección del modelo de regresión.**
14. **Evaluación de los supuestos de los residuales** del modelo de regresión elegido.

**Parte III — Comparación final, prueba y pronóstico**

15. **Comparación de los mejores modelos:** mejor modelo de cada familia en la validación y elección del modelo final.
16. **Evaluación en la muestra de prueba:** desempeño de los modelos finalistas en enero a junio de 2019.
17. **Pronóstico de los próximos 6 meses:** pronóstico de julio a diciembre de 2019 con sus intervalos de predicción.
18. **Conclusiones y limitaciones.**

## 1. Carga de paquetes

El notebook se desarrolló con Python 3.13 y las versiones de las librerías que se instalan en la celda siguiente. En un entorno que ya las tenga no hace falta instalarlas; en Google Colab, la celda de instalación se descomenta y se reinicia la sesión antes de continuar.

Después de cargar los paquetes se imprimen las versiones del entorno. Los modelos de suavización exponencial se estiman por máxima verosimilitud con un optimizador numérico, por lo que sus parámetros y errores pueden variar levemente entre sistemas operativos o procesadores, aun con las mismas versiones de las librerías. La regresión por mínimos cuadrados, en cambio, tiene una solución cerrada, por lo que sus coeficientes y errores deberían coincidir entre sistemas, salvo diferencias en los últimos decimales.

In [2]:
# Instalación de las librerías con las versiones usadas en este notebook.
# Solo es necesaria en Google Colab o en un entorno que no las tenga instaladas:
# quitar el comentario de la línea %pip y, en Colab, reiniciar la sesión después
# de instalar (Entorno de ejecución > Reiniciar sesión) para que se carguen las
# versiones nuevas. Requiere Python 3.12 o superior.

# %pip install pandas==3.1.1 numpy==2.5.4 matplotlib==3.11.2 statsmodels==0.15.0 scikit-learn==1.9.1 scipy==1.18.1 openpyxl==3.1.5

In [3]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Validación y métricas
from sklearn.model_selection import TimeSeriesSplit                # validación con origen móvil
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Descripción de la serie y modelos
from statsmodels.tsa.seasonal import seasonal_decompose            # descomposición (solo para describir la serie)
from statsmodels.tsa.exponential_smoothing.ets import ETSModel     # suavización exponencial (SES, Holt, Holt-Winters)
import statsmodels.api as sm                                       # regresión lineal por MCO y gráfico Q-Q

# Supuestos de los residuales de la regresión
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch  # autocorrelación y efectos ARCH
from statsmodels.sandbox.stats.runs import runstest_1samp          # prueba de rachas
from statsmodels.graphics.tsaplots import plot_acf                 # ACF de los residuales
from scipy import stats                                            # Jarque-Bera, Shapiro-Wilk y curva normal

In [4]:
# Versiones del entorno de ejecución, para poder reproducir los resultados
import platform
import sys

import matplotlib
import openpyxl
import scipy
import sklearn
import statsmodels

print(f"{'Python':<14}{sys.version.split()[0]}")
print(f"{'Plataforma':<14}{platform.platform()}")
for nombre_libreria, modulo in [
    ("pandas", pd),
    ("numpy", np),
    ("matplotlib", matplotlib),
    ("statsmodels", statsmodels),
    ("scikit-learn", sklearn),
    ("scipy", scipy),
    ("openpyxl", openpyxl),
]:
    print(f"{nombre_libreria:<14}{modulo.__version__}")

Python        3.13.9
Plataforma    macOS-27.0.1-arm64-arm-64bit-Mach-O
pandas        3.1.1
numpy         2.5.4
matplotlib    3.11.2
statsmodels   0.15.0
scikit-learn  1.9.1
scipy         1.18.1
openpyxl      3.1.5
